# 1.自定义工具

## 1.1.基于tool描述工具

In [6]:
from langchain_core.tools import tool


@tool("Square_root", description="Calculate the square root of a number")
def square_root(x :float) -> float:
    return x ** 0.5

## 1.2.使用函数名和文档注释描述工具

In [ ]:
@tool()
def get_weather(location: str, units: str = "Celsius", include_forecast: bool = False) -> str:
    """ Get the weather for a given location.
        Args:
            location (str): The location to get the weather for.
            units (str): The units to use for the temperature.
            include_forecast (bool): Whether to include the forecast.
    """
    # 采用硬编码返回数据，方便测试。现实开发应该调用天气API
    temp = 22 if units == "Celsius" else 72
    result = f"Current weather in {location} : {temp} degrees {units[0].upper()}"
    if include_forecast:
        result +="\nNext 5 days : sunny"
    return result

## 1.3.使用Pydantic Model描述参数

### · 当参数过多且需要描述时，采用pydantic model

In [8]:
from pydantic import BaseModel, Field
from typing import Literal

class WeatherParams(BaseModel):
    """查询天气输入的参数"""
    location: str = Field(description="City name or coordinates")
    units: Literal["Celsius", "Fahrenheit"] = Field(
        description="Units to use for temperature",
        default="Celsius"
    )
    include_forecast: bool = Field(
        description="Whether to include the 5-days forecast",
        default=False
    )

@tool(args_schema=WeatherParams)
def get_weather(location: str, units: str = "Celsius", include_forecast: bool = False) -> str:
    """ Get the weather for a given location."""
    temp = 22 if units == "Celsius" else 72
    result = f"Current weather in {location} : {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days : sunny"
    return result

#### · 工具调用方式和函数一样

In [10]:
square_root.invoke({"x" : 25})

5.0

#### · 测试

In [11]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(model="deepseek-chat",
        tools=[square_root, get_weather]
)

for token,metadata in agent.stream(
    {"messages" : [HumanMessage(content="今天杭州的天气怎么样？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

Current weather in 杭州 : 22 degrees C杭州今天当前气温是 **22°C**，不过我还只查到了实时温度这一项信息。需要我帮你查一下未来几天的预报吗？

In [12]:
response = agent.invoke(
    {"messages" : [HumanMessage(content="49和144的平方根是多少？")]}
)

for message in response['messages']:
    print(message.pretty_print())

================================ Human Message =================================

49和144的平方根是多少？
None
================================== Ai Message ==================================
Tool Calls:
  Square_root (call_00_RtpyNxfXpfq6Mcg6jB5Q9957)
 Call ID: call_00_RtpyNxfXpfq6Mcg6jB5Q9957
  Args:
    x: 49
  Square_root (call_01_H8lELHpTb6tqVKDjcoWH9158)
 Call ID: call_01_H8lELHpTb6tqVKDjcoWH9158
  Args:
    x: 144
None
================================= Tool Message =================================
Name: Square_root

7.0
None
================================= Tool Message =================================
Name: Square_root

12.0
None
================================== Ai Message ==================================

- 49 的平方根是 **7**
- 144 的平方根是 **12**
None
